# 第10章: イベントカメラSLAM - イベントカメラの基礎

**SLAM Handbook Chapter 10, Section 10.1-10.3**

このノートブックでは、イベントカメラの動作原理とSLAMへの応用の基礎を学びます。

## 学習内容
1. イベント生成モデル（Eq 10.1）
2. イベント蓄積とタイムサーフェス
3. モーション補償イベント画像
4. フレームベースカメラとの比較

イベントカメラは各ピクセルが独立に輝度変化を検出する非同期センサーであり、高速・高ダイナミックレンジ・低消費電力という特徴があります。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 10.1 イベント生成モデル

**SLAM Handbook Section 10.1, Eq 10.1**

イベントカメラの各ピクセル $(x, y)$ は、対数輝度 $L = \log(I)$ の変化がしきい値 $C$ を超えたときにイベントを生成します：

$$\Delta L(x, y, t) = L(x, y, t) - L(x, y, t - \Delta t) = p_k \cdot C$$

ここで：
- $p_k \in \{+1, -1\}$: イベントの極性（明→暗、暗→明）
- $C > 0$: コントラストしきい値（通常 0.1〜0.5）
- 各イベントは $(x_k, y_k, t_k, p_k)$ のタプルで表現

この非同期的な動作により、静止部分ではイベントが発生せず、動きのあるエッジ部分のみが検出されます。

In [ ]:
def create_test_scene(W=128, H=128):
    """Create a simple scene: white rectangle on black background."""
    scene = np.zeros((H, W), dtype=np.float64)
    # Rectangle
    scene[30:90, 20:50] = 200.0
    # Circle
    yy, xx = np.ogrid[:H, :W]
    circle_mask = (xx - 90)**2 + (yy - 60)**2 < 20**2
    scene[circle_mask] = 150.0
    # Gradient bar
    scene[100:115, 20:110] = np.linspace(50, 250, 90)
    return scene

def simulate_events(scene, velocity, dt=1e-3, n_steps=100, C=0.3, noise_rate=0.001):
    """
    Simulate event camera output for a translating scene.

    Parameters:
        scene: 2D intensity image
        velocity: [vx, vy] in pixels/second
        dt: time step
        n_steps: number of time steps
        C: contrast threshold (Eq 10.1: delta_L = p_k * C)
        noise_rate: random noise event probability per pixel per step
    """
    H, W = scene.shape
    events = []  # List of (x, y, t, p)

    # Initialize log intensity reference
    log_ref = np.log(scene + 1.0)  # +1 to avoid log(0)

    for step in range(1, n_steps + 1):
        t = step * dt
        # Shift scene by velocity * t
        shift_x = velocity[0] * t
        shift_y = velocity[1] * t

        # Create shifted scene via interpolation
        yy, xx = np.mgrid[:H, :W]
        src_x = (xx - shift_x).astype(np.float64)
        src_y = (yy - shift_y).astype(np.float64)

        # Simple nearest-neighbor lookup
        src_xi = np.clip(np.round(src_x).astype(int), 0, W - 1)
        src_yi = np.clip(np.round(src_y).astype(int), 0, H - 1)
        shifted_scene = scene[src_yi, src_xi]

        log_current = np.log(shifted_scene + 1.0)
        delta_L = log_current - log_ref

        # Generate events where |delta_L| > C
        pos_mask = delta_L >= C
        neg_mask = delta_L <= -C

        # Positive events (brightness increase)
        pos_coords = np.argwhere(pos_mask)
        for coord in pos_coords:
            events.append((coord[1], coord[0], t, +1))
        # Update reference for triggered pixels
        log_ref[pos_mask] += C

        # Negative events (brightness decrease)
        neg_coords = np.argwhere(neg_mask)
        for coord in neg_coords:
            events.append((coord[1], coord[0], t, -1))
        log_ref[neg_mask] -= C

        # Noise events
        noise_mask = np.random.random((H, W)) < noise_rate
        noise_coords = np.argwhere(noise_mask)
        for coord in noise_coords:
            p = np.random.choice([-1, +1])
            events.append((coord[1], coord[0], t, p))

    return np.array(events) if events else np.empty((0, 4))

# Create scene and simulate events
scene = create_test_scene()
velocity = [100.0, 30.0]  # pixels/second
events = simulate_events(scene, velocity, dt=1e-3, n_steps=80, C=0.2)

print(f"Total events generated: {len(events)}")
print(f"Positive events: {np.sum(events[:, 3] > 0)}")
print(f"Negative events: {np.sum(events[:, 3] < 0)}")
print(f"Time span: {events[:, 2].min():.3f} - {events[:, 2].max():.3f} s")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Original scene
axes[0].imshow(scene, cmap='gray')
axes[0].set_title('Original Scene (Intensity)')
axes[0].set_xlabel('x [px]')
axes[0].set_ylabel('y [px]')

# All events colored by polarity
ax = axes[1]
pos = events[events[:, 3] > 0]
neg = events[events[:, 3] < 0]
ax.scatter(neg[:, 0], neg[:, 1], s=0.1, c='blue', alpha=0.3, label=f'OFF ({len(neg)})')
ax.scatter(pos[:, 0], pos[:, 1], s=0.1, c='red', alpha=0.3, label=f'ON ({len(pos)})')
ax.set_xlim(0, 128)
ax.set_ylim(128, 0)
ax.set_aspect('equal')
ax.legend(markerscale=20)
ax.set_title('Event Stream (all events)')
ax.set_xlabel('x [px]')
ax.set_ylabel('y [px]')

# Events colored by time
ax = axes[2]
sc = ax.scatter(events[:, 0], events[:, 1], s=0.2, c=events[:, 2], cmap='viridis', alpha=0.5)
ax.set_xlim(0, 128)
ax.set_ylim(128, 0)
ax.set_aspect('equal')
plt.colorbar(sc, ax=ax, label='Time [s]')
ax.set_title('Event Stream (colored by time)')
ax.set_xlabel('x [px]')
ax.set_ylabel('y [px]')

plt.tight_layout()
plt.show()

## 10.2 イベント蓄積とタイムサーフェス

**SLAM Handbook Section 10.2**

イベントストリームを画像として表現する代表的な方法：

### イベント蓄積画像（Event Accumulation Image）
一定時間窓内のイベントを蓄積して2D画像を生成：
$$E(x, y) = \sum_{k: t_k \in [t-\Delta T, t]} p_k$$

### タイムサーフェス（Time Surface / SAE: Surface of Active Events）
各ピクセルの最新イベント発生時刻を記録：
$$\mathcal{T}(x, y) = \max_{k: (x_k, y_k) = (x, y)} t_k$$

タイムサーフェスは、エッジの最新位置と形状を滑らかに表現できるため、特徴追跡やオプティカルフローに適しています。

In [ ]:
def event_accumulation_image(events, W=128, H=128, t_start=None, t_end=None):
    """Create event accumulation image from events in [t_start, t_end]."""
    if t_start is None:
        t_start = events[:, 2].min()
    if t_end is None:
        t_end = events[:, 2].max()

    mask = (events[:, 2] >= t_start) & (events[:, 2] <= t_end)
    selected = events[mask]

    acc = np.zeros((H, W))
    for e in selected:
        x, y, t, p = int(e[0]), int(e[1]), e[2], e[3]
        if 0 <= x < W and 0 <= y < H:
            acc[y, x] += p
    return acc

def time_surface(events, W=128, H=128, t_ref=None, tau=0.01):
    """
    Create exponentially decaying time surface.
    T(x,y) = exp(-(t_ref - t_last(x,y)) / tau)
    """
    if t_ref is None:
        t_ref = events[:, 2].max()

    ts_pos = np.zeros((H, W))  # Time of last positive event
    ts_neg = np.zeros((H, W))  # Time of last negative event

    for e in events:
        x, y, t, p = int(e[0]), int(e[1]), e[2], e[3]
        if 0 <= x < W and 0 <= y < H:
            if p > 0:
                ts_pos[y, x] = max(ts_pos[y, x], t)
            else:
                ts_neg[y, x] = max(ts_neg[y, x], t)

    # Apply exponential decay
    surface_pos = np.exp(-(t_ref - ts_pos) / tau)
    surface_neg = np.exp(-(t_ref - ts_neg) / tau)
    surface_pos[ts_pos == 0] = 0
    surface_neg[ts_neg == 0] = 0

    return surface_pos, surface_neg

# Compute representations
acc_img = event_accumulation_image(events)
ts_pos, ts_neg = time_surface(events, tau=0.01)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Event accumulation
ax = axes[0]
vmax = max(abs(acc_img.min()), abs(acc_img.max()))
if vmax > 0:
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    im = ax.imshow(acc_img, cmap='RdBu_r', norm=norm)
else:
    im = ax.imshow(acc_img, cmap='RdBu_r')
plt.colorbar(im, ax=ax, label='Accumulated polarity')
ax.set_title('Event Accumulation Image\n赤=ON, 青=OFF')

# Time surface (positive)
ax = axes[1]
im = ax.imshow(ts_pos, cmap='hot')
plt.colorbar(im, ax=ax, label='Decay value')
ax.set_title('Time Surface (ON events)\nタイムサーフェス（正極性）')

# Time surface (negative)
ax = axes[2]
im = ax.imshow(ts_neg, cmap='hot')
plt.colorbar(im, ax=ax, label='Decay value')
ax.set_title('Time Surface (OFF events)\nタイムサーフェス（負極性）')

for ax in axes:
    ax.set_xlabel('x [px]')
    ax.set_ylabel('y [px]')

plt.tight_layout()
plt.show()

## 10.3 モーション補償イベント画像

**SLAM Handbook Section 10.3**

イベントカメラの動きが既知の場合、各イベントを参照時刻に逆投影（ワープ）することで、鮮明なエッジ画像を復元できます。

モーション補償の基本的な考え方：
- カメラの動きにより、同じエッジ上のイベントが異なるピクセル位置で発生
- 正しいモーションパラメータで補償すると、イベントが同じ位置に集約
- **Contrast Maximization**: 補償後の画像のコントラスト（分散）を最大化するモーションを探索

$$\mathbf{x}'_k = \mathbf{x}_k - (t_k - t_{ref}) \cdot \mathbf{v}$$

ここで $\mathbf{v}$ はオプティカルフロー（速度場）です。

In [ ]:
def motion_compensated_image(events, vx, vy, W=128, H=128, t_ref=None):
    """
    Warp events to reference time using motion model.
    x'_k = x_k - (t_k - t_ref) * vx
    y'_k = y_k - (t_k - t_ref) * vy
    """
    if t_ref is None:
        t_ref = events[:, 2].mean()

    img = np.zeros((H, W))
    for e in events:
        x, y, t, p = e[0], e[1], e[2], e[3]
        # Warp to reference time
        x_w = x - (t - t_ref) * vx
        y_w = y - (t - t_ref) * vy
        xi, yi = int(round(x_w)), int(round(y_w))
        if 0 <= xi < W and 0 <= yi < H:
            img[yi, xi] += 1
    return img

def compute_contrast(img):
    """Compute image contrast (variance of non-zero pixels)."""
    nonzero = img[img > 0]
    if len(nonzero) < 2:
        return 0
    return np.var(nonzero)

# Search for best motion compensation
vx_range = np.linspace(-150, 150, 61)
vy_range = np.linspace(-80, 80, 33)
contrast_map = np.zeros((len(vy_range), len(vx_range)))

for i, vy_test in enumerate(vy_range):
    for j, vx_test in enumerate(vx_range):
        img = motion_compensated_image(events, vx_test, vy_test)
        contrast_map[i, j] = compute_contrast(img)

# Find best velocity
best_idx = np.unravel_index(np.argmax(contrast_map), contrast_map.shape)
best_vy = vy_range[best_idx[0]]
best_vx = vx_range[best_idx[1]]

print(f"True velocity:      vx={velocity[0]:.1f}, vy={velocity[1]:.1f} px/s")
print(f"Estimated velocity: vx={best_vx:.1f}, vy={best_vy:.1f} px/s")

# Generate images for comparison
img_no_comp = motion_compensated_image(events, 0, 0)
img_correct = motion_compensated_image(events, best_vx, best_vy)
img_wrong = motion_compensated_image(events, -50, 50)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# No compensation
axes[0, 0].imshow(img_no_comp, cmap='hot')
axes[0, 0].set_title(f'No Motion Compensation\nContrast={compute_contrast(img_no_comp):.1f}')

# Correct compensation
axes[0, 1].imshow(img_correct, cmap='hot')
axes[0, 1].set_title(f'Best Compensation (vx={best_vx:.0f}, vy={best_vy:.0f})\nContrast={compute_contrast(img_correct):.1f}')

# Wrong compensation
axes[1, 0].imshow(img_wrong, cmap='hot')
axes[1, 0].set_title(f'Wrong Compensation (vx=-50, vy=50)\nContrast={compute_contrast(img_wrong):.1f}')

# Contrast map
ax = axes[1, 1]
im = ax.imshow(contrast_map, origin='lower', cmap='viridis', aspect='auto',
               extent=[vx_range[0], vx_range[-1], vy_range[0], vy_range[-1]])
ax.plot(velocity[0], velocity[1], 'r*', markersize=15, label='True')
ax.plot(best_vx, best_vy, 'wx', markersize=12, markeredgewidth=2, label='Estimated')
plt.colorbar(im, ax=ax, label='Contrast (Variance)')
ax.set_xlabel('vx [px/s]')
ax.set_ylabel('vy [px/s]')
ax.set_title('Contrast Maximization Map')
ax.legend()

for ax in axes.flat[:3]:
    ax.set_xlabel('x [px]')
    ax.set_ylabel('y [px]')

plt.tight_layout()
plt.show()

## 10.4 フレームベースカメラとの比較

| 特性 | フレームカメラ | イベントカメラ |
|------|-------------|-------------|
| 出力形式 | 同期フレーム画像 | 非同期イベントストリーム |
| 時間分解能 | ~30-60 fps | ~1 us（マイクロ秒） |
| ダイナミックレンジ | ~60 dB | ~120 dB |
| モーションブラー | あり | なし |
| 冗長データ | 多い（静止領域も記録） | 少ない（変化のみ） |
| 消費電力 | 高 | 低 |
| 既存アルゴリズム | 豊富 | 発展途上 |

In [ ]:
# Compare frame camera vs event camera for fast motion
def simulate_frame_camera(scene, velocity, exposure_time=0.033, n_samples=10):
    """Simulate a frame camera with motion blur."""
    H, W = scene.shape
    frame = np.zeros((H, W), dtype=np.float64)

    for i in range(n_samples):
        t = exposure_time * i / n_samples
        shift_x = velocity[0] * t
        shift_y = velocity[1] * t

        yy, xx = np.mgrid[:H, :W]
        src_xi = np.clip(np.round(xx - shift_x).astype(int), 0, W - 1)
        src_yi = np.clip(np.round(yy - shift_y).astype(int), 0, H - 1)
        frame += scene[src_yi, src_xi]

    return frame / n_samples

# Slow motion
velocity_slow = [10.0, 3.0]
events_slow = simulate_events(scene, velocity_slow, dt=1e-3, n_steps=80, C=0.2)
frame_slow = simulate_frame_camera(scene, velocity_slow)
acc_slow = event_accumulation_image(events_slow) if len(events_slow) > 0 else np.zeros((128, 128))

# Fast motion
velocity_fast = [300.0, 100.0]
events_fast = simulate_events(scene, velocity_fast, dt=1e-3, n_steps=30, C=0.2)
frame_fast = simulate_frame_camera(scene, velocity_fast)
acc_fast = event_accumulation_image(events_fast) if len(events_fast) > 0 else np.zeros((128, 128))

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Slow motion
axes[0, 0].imshow(scene, cmap='gray')
axes[0, 0].set_title('Original Scene')
axes[0, 1].imshow(frame_slow, cmap='gray')
axes[0, 1].set_title(f'Frame Camera (v={velocity_slow[0]:.0f} px/s)\nMotion blur: mild')
axes[0, 2].imshow(np.abs(acc_slow), cmap='hot')
axes[0, 2].set_title(f'Event Camera (v={velocity_slow[0]:.0f} px/s)\n{len(events_slow)} events')

# Fast motion
axes[1, 0].imshow(scene, cmap='gray')
axes[1, 0].set_title('Original Scene')
axes[1, 1].imshow(frame_fast, cmap='gray')
axes[1, 1].set_title(f'Frame Camera (v={velocity_fast[0]:.0f} px/s)\nMotion blur: severe')
axes[1, 2].imshow(np.abs(acc_fast), cmap='hot')
axes[1, 2].set_title(f'Event Camera (v={velocity_fast[0]:.0f} px/s)\n{len(events_fast)} events')

for ax in axes.flat:
    ax.set_xlabel('x [px]')
    ax.set_ylabel('y [px]')

plt.suptitle('Frame Camera vs Event Camera: Motion Blur Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 演習問題

1. **しきい値の影響**: コントラストしきい値 $C$ を0.05, 0.2, 0.5と変化させたとき、生成されるイベント数とエッジの鮮明さがどう変化するか調べよ。
2. **回転モーション**: 並進ではなく回転運動（画像中心まわり）でイベントを生成し、モーション補償を回転モデルに拡張せよ。
3. **ノイズ耐性**: `noise_rate` を増加させたとき、Contrast Maximizationによる速度推定精度への影響を分析せよ。

## まとめ

本ノートブックでは以下を学びました：

1. **イベント生成モデル**: 対数輝度変化がしきい値を超えたときに非同期的にイベントが発生する仕組み（Eq 10.1: $\Delta L = p_k \cdot C$）
2. **イベント表現**: イベント蓄積画像とタイムサーフェスによる可視化手法
3. **モーション補償**: Contrast Maximizationフレームワークによるモーション推定と鮮明なエッジ画像の復元
4. **フレームカメラとの比較**: 高速運動時のモーションブラー耐性の優位性

イベントカメラは、高速・高ダイナミックレンジが要求される環境でのSLAMに大きな可能性を持っています。

**参考**: SLAM Handbook Chapter 10